In [11]:
# Exercise 01
import findspark
findspark.init()
from pyspark import SparkContext

sc = SparkContext(appName="Excercise01", master="spark://namenode:7077")
input_file = "/data/BattleCreekDec19_2019.txt"

# Process everything
word_counts = sc.textFile(input_file) \
                .flatMap(lambda line: line.split(" ")) \
                .filter(lambda word: word != "") \
                .map(lambda word: (word.lower().strip(), 1)) \
                .reduceByKey(lambda a, b: a + b) \
                .sortBy(lambda x: x, ascending=False) # Simplified sorting syntax

# Note: .collect() pulls everything into driver memory. 
# If the file is massive, this can cause an OutOfMemory error.
all_results = word_counts.collect()

print(f"Grand Total: {len(all_results)} unique words found.\n")

# Printing thousands of lines might lag your notebook; consider all_results[:100]
for word, count in all_results:
    print(f"'{word}': {count}")
sc.stop()

Grand Total: 3299 unique words found.

'…"': 1
'…': 53
'‘well,': 1
'‘darling,': 1
'zone,': 1
'zeros?': 1
'zero': 1
'yourself.': 1
'your': 60
'young': 3
'you?"': 2
'you?': 3
'you."': 4
'you.': 40
'you,': 22
'you've': 5
'you're': 40
'you'll': 3
'you'd': 3
'you': 392
'york,': 1
'york': 2
'yet,': 1
'yet': 3
'yesterday,': 2
'yesterday': 1
'yes.': 1
'yellow': 2
'years…': 1
'years."': 2
'years.': 26
'years,': 13
'years': 19
'year."': 1
'year.': 4
'year,': 3
'year': 16
'yeah,': 2
'wrong?"': 2
'wrong.': 3
'wrong,': 2
'wrong': 1
'write.': 2
'write,': 1
'write': 3
'wrist': 1
'wrapped': 1
'wow.': 1
'wow!': 1
'wouldn't': 8
'would've': 4
'would': 32
'worst': 5
'worse': 4
'worry': 7
'worried.': 1
'worried,': 1
'worried': 1
'world.': 7
'world,': 4
'world': 7
'working.': 1
'working': 1
'workers,': 1
'workers': 3
'worker.': 2
'worker,': 1
'worker': 1
'worked': 2
'work.': 3
'work,': 1
'work': 6
'wore': 1
'words,': 1
'words': 3
'wording,': 1
'word.': 4
'word': 5
'wonderful,': 2
'wonderful': 6
'wonder': 1


In [12]:
# Exercise 02
import findspark
findspark.init()
from pyspark import SparkContext

sc = SparkContext(appName="Excercise02", master="spark://namenode:7077")
input_file = "/data/BattleCreekDec19_2019.txt"
lines = sc.textFile(input_file)

# 2. Extract words and filter out empty strings
words = lines.flatMap(lambda line: line.split(" ")) \
             .filter(lambda word: word != "") \
             .map(lambda word: word.strip().strip('.,!?:;()')) \
             .filter(lambda word: word != "")

# 3. Map each word to (length, 1) to prepare for sum and count
# (word_length, count)
word_stats = words.map(lambda word: (len(word), 1))

# 4. Reduce to get total length and total word count
# aggregate = (total_length, total_word_count)
total_length, total_count = word_stats.reduce(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# 5. Calculate average
if total_count > 0:
    average_length = total_length / total_count
    print(f"Total Words: {total_count}")
    print(f"Total Characters: {total_length}")
    print(f"Average Word Length: {average_length:.2f}")
else:
    print("No words found.")

sc.stop()

Total Words: 17830
Total Characters: 74715
Average Word Length: 4.19


In [16]:
# Exercise 03
import findspark
findspark.init()
from pyspark import SparkContext

sc = SparkContext(appName="Excercise03", master="spark://namenode:7077")
input_file = "/data/BattleCreekDec19_2019.txt"

counts = sc.textFile(input_file) \
           .flatMap(lambda line: line.split(" ")) \
           .filter(lambda word: word != "") \
           .map(lambda word: (word.lower().strip(), 1)) \
           .reduceByKey(lambda a, b: a + b)

for word, count in counts.takeOrdered(10, key=lambda x: -x[1]):
    print(f"'{word}': {count}")

sc.stop()

'the': 698
'and': 494
'i': 491
'to': 422
'you': 392
'a': 361
'they': 316
'of': 308
'we': 251
'in': 196


In [17]:
# Exercise 04
import findspark
findspark.init()
from pyspark import SparkContext

sc = SparkContext(appName="Excercise04", master="spark://namenode:7077")

# Define paths (using /data/ as per your notebook convention)
path1 = "/data/BattleCreekDec19_2019.txt"
path2 = "/data/BemidjiSep18_2020.txt"

# Helper function for word count RDD
def get_word_counts(file_path):
    return sc.textFile(file_path) \
             .flatMap(lambda line: line.split(" ")) \
             .map(lambda word: word.lower().strip().strip('.,!?:;()')) \
             .filter(lambda word: word != "") \
             .map(lambda word: (word, 1)) \
             .reduceByKey(lambda a, b: a + b)

# Create two RDDs of (word, count)
rdd1 = get_word_counts(path1)
rdd2 = get_word_counts(path2)

# Perform Inner Join
# Result format: (word, (count_from_file1, count_from_file2))
joined_rdd = rdd1.join(rdd2)

# Sort by combined frequency for display
sorted_joined = joined_rdd.sortBy(lambda x: -(x[1][0] + x[1][1]))

# Collect ALL results to the driver
all_joined_results = sorted_joined.collect()

print(f"Inner Join Results: {len(all_joined_results)} words present in BOTH files")
print("-" * 50)
for word, counts in all_joined_results:
    count1, count2 = counts
    print(f"'{word}': BattleCreek={count1}, Bemidji={count2}")

sc.stop()

Inner Join Results: 1072 words present in BOTH files
--------------------------------------------------
'the': BattleCreek=698, Bemidji=612
'and': BattleCreek=494, Bemidji=488
'i': BattleCreek=491, Bemidji=459
'to': BattleCreek=427, Bemidji=402
'you': BattleCreek=457, Bemidji=346
'a': BattleCreek=362, Bemidji=430
'they': BattleCreek=316, Bemidji=292
'of': BattleCreek=311, Bemidji=272
'it': BattleCreek=267, Bemidji=296
'that': BattleCreek=253, Bemidji=250
'we': BattleCreek=251, Bemidji=204
'in': BattleCreek=209, Bemidji=177
'have': BattleCreek=189, Bemidji=170
'but': BattleCreek=165, Bemidji=193
'he': BattleCreek=111, Bemidji=166
'it's': BattleCreek=141, Bemidji=134
'so': BattleCreek=148, Bemidji=126
'said': BattleCreek=142, Bemidji=131
'was': BattleCreek=120, Bemidji=144
'is': BattleCreek=118, Bemidji=133
'do': BattleCreek=126, Bemidji=108
'know': BattleCreek=136, Bemidji=95
'what': BattleCreek=150, Bemidji=78
'don't': BattleCreek=120, Bemidji=100
'people': BattleCreek=110, Bemidji=100

In [15]:
# Exercise 05
import findspark
findspark.init()
from pyspark import SparkContext

sc = SparkContext(appName="Excercise05", master="spark://namenode:7077")
input_file = "/data/BattleCreekDec19_2019.txt"

# 1. Load lines and split into words
raw_words = sc.textFile(input_file) \
              .flatMap(lambda line: line.split(" ")) \
              .map(lambda word: word.lower().strip().strip('.,!?:;()')) \
              .filter(lambda word: word != "")

# 2. Get unique words using .distinct()
unique_words_rdd = raw_words.distinct()

# 3. Count the results
total_unique = unique_words_rdd.count()

print(f"Total words (including duplicates): {raw_words.count()}")
print(f"Total unique words (duplicates removed): {total_unique}")

# Show a sample of 10 unique words
print("\nSample of unique words:")
print(unique_words_rdd.take(10))

sc.stop()

Total words (including duplicates): 17830
Total unique words (duplicates removed): 2356

Sample of unique words:
['vice', 'good', 'job', 'merry', 'christmas', 'michigan', 'we', 'in', 'was', 'of']
